TASK1


In [ ]:
!pip install nltk

In [ ]:
import nltk

In [ ]:
nltk.download("all")

[nltk_data] Downloading collection 'all'
[nltk_data]    | 
[nltk_data]    | Downloading package abc to /root/nltk_data...
[nltk_data]    |   Package abc is already up-to-date!
[nltk_data]    | Downloading package alpino to /root/nltk_data...
[nltk_data]    |   Package alpino is already up-to-date!
[nltk_data]    | Downloading package averaged_perceptron_tagger to
[nltk_data]    |     /root/nltk_data...
[nltk_data]    |   Package averaged_perceptron_tagger is already up-
[nltk_data]    |       to-date!
[nltk_data]    | Downloading package averaged_perceptron_tagger_eng to
[nltk_data]    |     /root/nltk_data...
[nltk_data]    |   Package averaged_perceptron_tagger_eng is already
[nltk_data]    |       up-to-date!
[nltk_data]    | Downloading package averaged_perceptron_tagger_ru to
[nltk_data]    |     /root/nltk_data...
[nltk_data]    |   Package averaged_perceptron_tagger_ru is already
[nltk_data]    |       up-to-date!
[nltk_data]    | Downloading package averaged_perceptron_tagger_r

True

In [ ]:
nltk.download('punkt')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [ ]:
pip install scikit-learn

In [ ]:
import pandas as pd
import numpy as np
import string
import re
from  nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import f1_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt

In [ ]:
dataset = pd.read_csv("IMDB Dataset.csv")
print(dataset)

                                                  review sentiment
0      One of the other reviewers has mentioned that ...  positive
1      A wonderful little production. <br /><br />The...  positive
2      I thought this was a wonderful way to spend ti...  positive
3      Basically there's a family where a little boy ...  negative
4      Petter Mattei's "Love in the Time of Money" is...  positive
...                                                  ...       ...
49995  I thought this movie did a down right good job...  positive
49996  Bad plot, bad dialogue, bad acting, idiotic di...  negative
49997  I am a Catholic taught in parochial elementary...  negative
49998  I'm going to have to disagree with the previou...  negative
49999  No one expects the Star Trek movies to be high...  negative

[50000 rows x 2 columns]


In [ ]:
#TOKENIZING
from nltk.tokenize import word_tokenize
dataset['tokenized_review'] = dataset['review'].apply(word_tokenize)
print(dataset['tokenized_review'].head())

In [ ]:
#prepocessing
dataset["review"] = dataset["tokenized_review"].apply(lambda x: ' '.join(x) if isinstance(x, list) else x)
print(dataset["review"].head())

In [ ]:
#TF-IDF
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(dataset["review"])
y = dataset["sentiment"]

#split train-test data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify = y)

#model training
model = MultinomialNB()
model.fit(X_train, y_train)

#predictions
y_predict = model.predict(X_test)
print(y_predict)

#performance
f1 = f1_score(y_test, y_predict, pos_label = "positive")
print(f"F1 Score: {f1:.4f}")

cm = confusion_matrix(y_test, y_predict, labels = ["negative", "positive"])
print("Confusion Matrix:")
print(cm)
disp = ConfusionMatrixDisplay(confusion_matrix = cm, display_labels = ["negative", "positive"])
disp.plot(cmap=plt.cm.Blues)
plt.title("Confusion Matrix")
plt.show()

c_report = classification_report(y_test, y_predict, target_names = ["negative", "positive"])
print("\nClassification Report:", c_report)

TASK2


In [ ]:
#GloVe embeddings

with open('glove.6B.100d.txt', 'r', encoding='utf-8') as file:
    glove_embeddings = {}
    for line in file:
        values = line.split()
        word = values[0]
        vector = np.array(values[1:], dtype='float32')
        glove_embeddings[word] = vector

def similarity(word1, word2):
    if word1 not in glove_embeddings or word2 not in glove_embeddings:
      return None

    v1 = glove_embeddings[word1]
    v2 = glove_embeddings[word2]
    similarity = np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))
    return similarity


In [ ]:
def analogy(word1, word2, word3, embeddings):
  if word1 not in embeddings or word2 not in embeddings or word3 not in embeddings:
    return None

  vec1 = embeddings[word1]
  vec2 = embeddings[word2]
  vec3 = embeddings[word3]

  analogy_vector = vec1 - vec2 + vec3

  closest_word = None
  closest_similarity = -1

  for word, vector in embeddings.items():
    if word in [word1, word2, word3]:
      continue

    similarity = np.dot(analogy_vector, vector) / (np.linalg.norm(analogy_vector) * np.linalg.norm(vector))

    if similarity > closest_similarity:
      closest_similarity = similarity
      closest_word = word

  return closest_word

# Perform the analogy task
result_word = analogy("paris", "france", "italy", glove_embeddings)

if result_word:
  print(f"'{result_word}' is the closest word to 'Paris - France + Italy'")
else:
  print("One or more words not found in the embeddings.")

TASK3


In [ ]:
pip install gensim

In [ ]:
from collections import Counter
from gensim.models import Word2Vec
from sklearn.decomposition import PCA

In [ ]:
nltk.download("gutenberg")

In [ ]:
from nltk.corpus import gutenberg

In [ ]:
books = gutenberg.fileids()
texts = " ".join([gutenberg.raw(fileid) for fileid in books])
tokens = word_tokenize(texts)
tokens = [word.lower() for word in tokens if word.isalpha()]
print(tokens)

In [ ]:
from nltk.tokenize import sent_tokenize
sentences = sent_tokenize(texts)
print(sentences)

tokenized_sentences = [word_tokenize(sent) for sent in sentences]
tokenized_sentences = [[word.lower() for word in sent if word.isalpha()] for sent in tokenized_sentences]
print(tokenized_sentences)


In [ ]:
#skipgram
model = Word2Vec(sentences=tokenized_sentences, vector_size=100, window=5, min_count=1, sg=1)
#cbow
model = Word2Vec(sentences=tokenized_sentences, vector_size=100, window=5, min_count=1, sg=0)


In [ ]:
#skipgram model's word similarity


In [ ]:
#cbow model's word similarity

In [ ]:
# Evaluate Skipgram model on analogy task (Paris - France + Italy)
# Evaluate CBOW model on analogy task (Paris - France + Italy)
# Apply PCA to reduce the dimensionality
# Plot the 2D PCA projection for Skipgram model
# Plot the 2D PCA projection for CBOW model